# Running the campaign, one batch at a time

This notebook executes one batch of the IBM TN-VQE campaign through Cebule's
`TN_QC_OPT` task. It is deliberately one batch per session: each batch maps
to a separate IBM access-plan purchase, and mixing them makes the spend
against a given plan impossible to read afterwards.

The campaign itself, and the reasoning behind what each run does, is in
[the campaign README](README.md). The batch files are generated by
[`ibm_tn_vqe_campaign.ipynb`](ibm_tn_vqe_campaign.ipynb) and are inputs
here, never rewritten.

**Three things protect purchased time.** Nothing submits until `SUBMIT` is
set to `True`, so a stray "Run All" costs nothing. Every completed run is
checkpointed, so re-running the notebook resumes rather than re-spending.
And the loop stops if cumulative estimated spend would exceed the batch's
plan budget.

Credentials are read from the environment and never entered here. Set
`CEBULE_EMAIL` and `CEBULE_PASSWORD` in the repo's `.env` (copy
`.env.example`), or export them in the shell that launches Jupyter.

In [ ]:
BATCH = "batch0_classical_only"   # see BATCHES below
SUBMIT = False                    # set True to actually submit
STOP_AFTER = None                 # e.g. 1 to submit a single run, then stop

# None means "use each run's own Backend_Platform column", which is what you
# want almost always: stage 0 crosses the backend as a factor, half its runs
# on aer_simulator and half on fake_aachen, so naming one here would run one
# arm twice and the other never. Set it only to execute a hardware-targeted
# stage-1 run on a simulator first.
BACKEND_OVERRIDE = None           # or "aer_simulator" / "fake_aachen"

# A guard rather than a plan budget: set it to stop a batch before it
# exceeds a number of QPU minutes you chose. None means run the batch.
BUDGET_MIN = None

# Batch order is deliberate, and it is an ORDER rather than a set of
# budgets. The campaign holds one 900-minute allocation, not a sequence of
# separate IBM plan purchases, so nothing here caps a batch: stage 0 costs
# no QPU time at all, batch0 is the classical-only control, batch1 is the
# single cheapest hardware run and exists to prove the submission path, and
# batch2 is the rest of the stage-1 screen.
BATCHES = ["stage0_simulator_screen", "batch0_classical_only",
           "batch1_pipeline_check", "batch2_screen"]
assert BATCH in BATCHES, f"unknown batch {BATCH!r}"

# Stage 0 is 1008 runs. This notebook runs a batch whole, so drive that one
# from utils/run_campaign.py instead, which slices it with --where filters
# and resumes:
#     PYTHONPATH=src python utils/run_campaign.py --group-by Molecule,Basis,Mapper
print(f"batch:    {BATCH}")
default = "each run's own Backend_Platform"
print(f"backend:  {BACKEND_OVERRIDE or default}")
print(f"submit:   {SUBMIT}"
      + ("" if SUBMIT else "   (dry run: inputs are built and shown, nothing is sent)"))


### Which backend, and in what order

The campaign runs every batch on both simulators before any purchased time is
committed, so `BACKEND` is the knob you turn last:

| `BACKEND` | `TNQCOptInput.backend` | What the pass establishes |
|---|---|---|
| `aer_simulator` | `aer_simulator` | The algorithmic result without device error, and the reference every hardware result is read against |
| `fake_aachen` | `fake_aachen` | The energy shift and change in convergence attributable to device error alone. An offline calibration snapshot of `ibm_aachen` itself, so it needs no credentials |
| `hardware` | the run's own `Backend_Platform` | The real measurement. The only setting that spends plan budget |

Results are checkpointed per batch *and* per backend, so the three passes do
not collide: a simulated run does not mark its hardware counterpart complete,
and each pass resumes independently.

A `network` run takes no quantum measurements at all, so it is always routed
to a simulator whatever this is set to. Naming hardware there would make
Cebule authenticate against a device the run never uses.

In [ ]:
import csv
import json
import os
import pathlib
import sys

CAMPAIGN = pathlib.Path.cwd()
while not (CAMPAIGN / "pyproject.toml").exists():
    CAMPAIGN = CAMPAIGN.parent
REPO = CAMPAIGN
CAMPAIGN = REPO / "data" / "benchmarks" / "ibm_tn-vqe_qesem"

sys.path.insert(0, str(REPO / "src"))
sys.path.insert(0, str(REPO / "utils"))


In [ ]:
# Everything that builds a run lives in utils/_campaign_runner.py, which
# utils/run_campaign.py uses too, so the notebook and the CLI cannot come
# to disagree about what a run is.
import _campaign_runner as runner

runs = list(csv.DictReader((CAMPAIGN / f"{BATCH}.csv").open()))
estimated_min = sum(float(r.get("Est_QPU_Time_S") or 0) for r in runs) / 60

# Read off the runs rather than declared: get_backend routes anything
# prefixed 'ibm' to real hardware, and a network run never reaches a device
# whatever its column says, so only the resolved set is the truth.
backends = {runner.backend_for(r, BACKEND_OVERRIDE) for r in runs}
on_hardware = any(b.startswith("ibm") for b in backends)
budget_min = BUDGET_MIN if on_hardware else None

print(f"{len(runs)} runs on {', '.join(sorted(backends))}"
      + ("   (PURCHASED QPU TIME)" if on_hardware else "   (simulated)"))
if estimated_min:
    print(f"estimated QPU time: {estimated_min:,.2f} min on hardware"
          + ("" if on_hardware else ", but this pass is simulated and spends none of it"))
else:
    print("no QPU time: these runs take no quantum measurements")


## Where results go

Each finished run is appended to an NDJSON file as a single line: the run's
identifying columns, the backend it ran on, the returned energy, the cost
history, and the wall-clock time it took. Wall-clock is recorded because the
campaign measures whether TN-VQE displaces quantum work onto CPU, and
`TNQCOptResult` returns no timing of its own.

The file is the checkpoint, and it is named for the batch and the backend
together. A run whose `Case_ID` already appears in it is skipped, so an
interrupted pass resumes where it stopped, while the same run on a different
backend is still outstanding.

In [ ]:
RESULTS = runner.RESULTS_DIR / f"{BATCH}.ndjson"
RESULTS.parent.mkdir(parents=True, exist_ok=True)

def completed_case_ids() -> set[str]:
    return runner.completed_case_ids(RESULTS)

done = completed_case_ids()
pending = [r for r in runs if r["Case_ID"] not in done]
print(f"{len(done)} completed, {len(pending)} pending -> {RESULTS.relative_to(REPO)}")


## Building one run's input

Three things are assembled per run, and each is checked rather than assumed.

The **circuit** comes from the committed OpenQASM file the run names, and its
SHA-256 is verified against the value recorded in the campaign. A mismatch
means the file changed after the campaign was generated, which would make
the run incomparable with the rest, so it raises rather than proceeding.
Supplying `qasm_ansatz` also changes `n_layers_circuit`'s effective default
from 3 to 1 upstream, so it is passed explicitly.

The **initial circuit parameters** are the campaign's pinned draw, keyed on
the ansatz alone so that runs sharing a circuit start from the same point.

The **Hamiltonian** comes from `hamiltonian_data/`, which holds the operator
each run optimises as dense Pauli strings. For mol_map runs that file is the
only source, since the constraint encoding is Cebule's and cannot be derived
here. Where no file exists, a Jordan-Wigner operator is built from the run's
own geometry, basis and active space; where neither is possible the run is
reported and skipped rather than silently mis-built.

Every file is checked against the run before use: operator count against
coefficient count, and operator width against the run's own `N_Qubit`. A
file describing a different system fails loudly instead of quietly
optimising the wrong Hamiltonian.

One mapping is worth stating, because the campaign's columns do not spell
it out. A plain-VQE run is `TN_QC_OPT` with the tensor network switched
off: `optimization_mode="circuit"` freezes `θ`, and zero network layers
means there is no `θ` to freeze. That is why `TN_Layers_Network` and
`TN_Ansatz` are empty on those runs, and why the three methods remain
comparable, since all three reach the QPU through the same task.

In [ ]:
import _campaign_runner as runner
from _campaign_runner import (
    build_input,
    unbuildable_reason,
)

# Build the first pending run to see what a submission looks like.
if pending:
    sample = build_input(pending[0], BACKEND_OVERRIDE)
    if sample is None:
        print(f"run {pending[0]['Case_ID']}: cannot be built here -- "
              f"{unbuildable_reason(pending[0])}")
    else:
        print(f"run {pending[0]['Case_ID']}: {runner.run_label(pending[0], BACKEND_OVERRIDE)}")
        print(f"  {len(sample.h_coeff_values)} Hamiltonian terms, "
              f"{len(sample.phi_init)} circuit parameters, "
              f"{sample.n_iterations} evaluations, backend {sample.backend}")


## Submitting

The session is opened once and reused. Credentials come from the
environment: this cell reads `CEBULE_EMAIL` and `CEBULE_PASSWORD` and never
prompts, so nothing is typed into a notebook that gets saved to disk. It
loads the repo's gitignored `.env` first, since Jupyter does not read that
file on its own; anything already exported in the shell takes precedence.

In [ ]:
session = runner.open_session() if SUBMIT else None
print(f"session opened as {os.environ['CEBULE_EMAIL']}" if SUBMIT else
      "SUBMIT is False: the loop below will build and validate, not send")


## The run loop

One run at a time, checkpointed after each. Before submitting anything the
loop re-reads the results file and skips every run already in it, so an
interrupted pass is resumed simply by running this cell again: nothing is
paid for twice, and the check is made per run rather than from a list built
earlier in the session.

The spend guard applies only on hardware, since a simulated pass consumes no
plan budget. It is on estimated time rather than billed, because billed is
only known after the fact; it is there to stop a runaway loop, not to account
precisely.

In [ ]:
# Re-read the checkpoint here rather than reusing the `pending` list built
# further up. This is the cell you re-run after an interruption, and running
# it alone would otherwise iterate a stale list and re-submit runs that are
# already in the results file. Iterating `runs` and testing membership makes
# the skip decision per run, at the moment it is made.
done = completed_case_ids()
spent_min = sum(float(r["Est_QPU_Time_S"] or 0) for r in runs if r["Case_ID"] in done) / 60
submitted = skipped_done = 0

for run in runs:
    case, estimate_min = run["Case_ID"], float(run.get("Est_QPU_Time_S") or 0) / 60

    if case in done:
        skipped_done += 1
        continue
    if budget_min is not None and spent_min + estimate_min > budget_min:
        print(f"stopping before run {case}: would take the batch to "
              f"{spent_min + estimate_min:,.1f} min of a {budget_min} min budget")
        break
    if STOP_AFTER is not None and submitted >= STOP_AFTER:
        print(f"stopping after {submitted} run(s), as STOP_AFTER asked")
        break

    task_input = build_input(run, BACKEND_OVERRIDE)
    if task_input is None:
        print(f"  skip {case}: {unbuildable_reason(run)}")
        continue

    label = runner.run_label(run, BACKEND_OVERRIDE)
    if not SUBMIT:
        print(f"  would submit {case}: {label}, {estimate_min:.2f} min")
        continue

    print(f"  submitting {case}: {label} ...", end=" ", flush=True)
    result, wall_s, task_id = runner.submit_run(
        session, run, task_input, f"{BATCH}-case{case}",
    )
    # Written and closed per run, so an interrupt loses at most the run in
    # flight. The next pass reads this file and skips everything in it.
    runner.append_record(
        RESULTS, run, result, task_id, wall_s, BATCH, BACKEND_OVERRIDE,
    )

    done.add(case)
    spent_min += estimate_min
    submitted += 1
    print(f"E = {result.vqe_energy:.6f} Ha, {wall_s:.1f} s wall clock")

print(f"\n{submitted} submitted this session, {skipped_done} already in the results "
      f"file and skipped, {len(done)} of {len(runs)} complete")


## Where the batch stands

Wall-clock time is summarised alongside the energies because it is the
classical half of the measurement: a `network` run consumes no QPU time at
all, so its wall clock is pure CPU cost, and comparing it against the
`both`-mode runs it controls is what quantifies the trade the method makes.

In [ ]:
if RESULTS.exists():
    records = [json.loads(line) for line in RESULTS.open() if line.strip()]
    print(f"{len(records)} of {len(runs)} runs complete\n")
    print(f"{'case':>5}  {'molecule':16}{'method':9}{'mode':9}{'energy (Ha)':>13}{'wall (s)':>10}")
    for rec in sorted(records, key=lambda r: int(r["Case_ID"])):
        print(f"{rec['Case_ID']:>5}  {rec['Molecule'] + '/' + rec['Basis']:16}"
              f"{rec['Method']:9}{rec['Optimization_Mode']:9}"
              f"{rec['vqe_energy']:>13.6f}{rec['wall_clock_s']:>10.1f}")
    total_wall = sum(r["wall_clock_s"] for r in records)
    print(f"\ntotal wall clock {total_wall / 60:,.1f} min, "
          f"estimated QPU {sum(r['estimated_qpu_s'] for r in records) / 60:,.1f} min")
else:
    print("nothing completed yet")

## Moving to the next batch

Set `BATCH` to the next entry and re-run. The order is deliberate:

- **`stage0_simulator_screen`** is 1008 runs on `aer_simulator` and
  `fake_aachen`. It spends none of the 900-minute allocation and never
  reaches hardware. It is what replaces the campaign's assumed measurement
  counts, evaluation budgets and ansatz/optimizer choices with measured
  ones, so it runs first. **Drive this one from the command line instead**,
  which slices it into batches and resumes:

  ```sh
  PYTHONPATH=src python utils/run_campaign.py --group-by Molecule,Basis,Mapper
  PYTHONPATH=src python utils/run_campaign.py --submit \
      --where Molecule=H2 --where Basis=6-31g --where Mapper=JW
  ```

- **`batch0_classical_only`** optimises `theta` by classical
  tensor-network contraction and takes no quantum measurements.
- **`batch1_pipeline_check`** is the single cheapest hardware run, and
  exists to prove the submission path end to end before purchased time is
  committed to anything else.
- **`batch2_screen`** is the rest of the stage-1 hardware screen.

Nothing here is capped to a plan budget. Set `BUDGET_MIN` in the first cell
if you want the loop to stop before a chosen number of QPU minutes.

Both this notebook and `utils/run_campaign.py` build their runs through
`utils/_campaign_runner.py`, so a correction to how a run is assembled
reaches both. The notebook runs a batch whole and shows its working; the
script slices, filters and resumes.

Two things stage 0 is expected to report back, both of which the campaign
currently assumes: the measurement count each Hamiltonian really groups
into, which is carried here as a greedy upper bound about twice the real
value, and the evaluation budget a run really needs to converge.
